# Exercise Solution — Transaction Monitor: Producer
Generates random banking transactions and streams them into the `transactions` Kafka topic.

In [ ]:
from kafka import KafkaProducer
from datetime import datetime
from random import uniform, choice
from time import sleep
import json

# ----- Configuration -----
TOPIC    = 'transactions'
BROKERS  = ['course-kafka:9092']

ACCOUNTS = ['ACC_1', 'ACC_2', 'ACC_3', 'ACC_4', 'ACC_5']
CITIES   = ['Tel Aviv', 'Jerusalem', 'Haifa', 'Beer Sheva']

In [ ]:
def transaction_generator():
    """
    Infinite generator that yields one random transaction dict at a time.
    Using a generator keeps the logic clean and memory-efficient —
    we never hold all transactions in memory at once.
    """
    while True:
        yield {
            'account_id': choice(ACCOUNTS),
            'amount':     round(uniform(10, 5000), 2),
            'city':       choice(CITIES),
            'timestamp':  datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        }

In [ ]:
# acks=1  → the broker leader confirms receipt before we continue (balanced durability)
# retries=3 → automatically retry up to 3 times on transient network errors
producer = KafkaProducer(
    bootstrap_servers = BROKERS,
    acks              = 1,
    retries           = 3
)

In [ ]:
for tx in transaction_generator():

    # Serialize the dict to a UTF-8 JSON byte string —
    # Kafka messages are always raw bytes, never Python objects.
    value = json.dumps(tx).encode('utf-8')

    # Use account_id as the key so that all transactions from the same account
    # always land in the same partition → guaranteed ordering per account.
    key = tx['account_id'].encode('utf-8')

    producer.send(topic=TOPIC, key=key, value=value)

    # flush() blocks until the message is confirmed by the broker.
    # Without this, the message lives only in an internal buffer and
    # could be lost if the process exits before the buffer is drained.
    producer.flush()

    print(f"Sent → {tx['account_id']} | {tx['amount']:.2f} ILS | {tx['city']}")

    # Simulate a real stream with a random delay between messages
    sleep(uniform(1, 3))